# 00.5 sklearn Minimal Pipeline / sklearn 最小建模流程

这份 notebook 的目标是建立一个非常小但完整的机器学习建模流程。  
The goal of this notebook is to build a very small but complete machine-learning workflow.

你后面学 `PyTorch` 时，会不断用到这里的思路：  
You will reuse the same ideas when you later learn `PyTorch`:

- 训练集 / 测试集划分 / train-test split
- 基线模型 / baseline model
- 标准化 / standardization
- 指标评估 / evaluation metrics
- 数据泄漏 / data leakage

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 用 `train_test_split` 切分数据 / Split data with `train_test_split`.
2. 解释为什么要先切分再标准化 / Explain why we split before scaling.
3. 训练一个基础分类模型 / Train a basic classifier.
4. 用准确率和混淆矩阵评估模型 / Evaluate with accuracy and a confusion matrix.
5. 理解基线模型的作用 / Understand the role of a baseline model.
6. 把这一流程迁移到后面的神经网络任务 / Transfer this workflow to later neural-network tasks.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. 读入数据 / Loading Data

这里使用 `iris` 数据集，因为它小、稳定、适合演示。  
We use the `iris` dataset because it is small, stable, and suitable for demonstration.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("特征前几行 / feature head:")
print(X.head())
print()
print("标签前几行 / target head:")
print(y.head())
print()
print("类别名称 / target names:", iris.target_names)

## 2. 训练集与测试集 / Train-Test Split

为什么要切分？  
Why do we split the data?

- 训练集 / training set: 用来学习参数 / used to learn model parameters
- 测试集 / test set: 用来评估泛化 / used to estimate generalization

如果你在测试集上调来调去，评估就会失真。  
If you keep tuning against the test set, your evaluation becomes biased.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 3. 基线模型 / Baseline Model

基线模型 / baseline model 的意义是：先有一个最低参考线。  
A baseline model gives you a minimum reference line before you try something more advanced.

如果你的复杂模型还不如基线模型，那通常说明流程有问题。  
If your complex model is not better than the baseline, the pipeline is usually the problem.

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_pred = dummy_clf.predict(X_test)
dummy_acc = accuracy_score(y_test, dummy_pred)

print("基线准确率 / baseline accuracy:", dummy_acc)

## 4. 标准化与 Pipeline / Standardization and Pipeline

很多模型受特征尺度影响，所以常要标准化 / standardize features.  
Many models are sensitive to feature scales, so standardization is common.

为什么建议用 `Pipeline`？  
Why use a `Pipeline`?

- 把预处理和模型绑定在一起 / binds preprocessing and the model together
- 降低数据泄漏风险 / reduces data leakage risk
- 代码更清晰 / keeps the code cleaner

In [ ]:
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500)),
    ]
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
acc = accuracy_score(y_test, pred)

print("逻辑回归准确率 / logistic regression accuracy:", acc)

注意顺序 / Important order:

1. 先切分 / split first
2. 再用训练集拟合标准化器 / fit the scaler on the training set
3. 再把同样的变换用到测试集 / apply the same transform to the test set

这就是避免数据泄漏 / data leakage 的关键之一。  
This is one of the key steps for avoiding data leakage.

## 5. 评估 / Evaluation

除了准确率 / accuracy，分类任务里还应关注：

- 混淆矩阵 / confusion matrix
- 分类报告 / classification report

因为单个准确率会隐藏错误结构。  
A single accuracy number can hide the structure of errors.

In [ ]:
cm = confusion_matrix(y_test, pred)
report = classification_report(y_test, pred, target_names=iris.target_names)

print("混淆矩阵 / confusion matrix:")
print(cm)
print()
print("分类报告 / classification report:")
print(report)

## 6. 和基线比较 / Comparing with the Baseline

真正有意义的不是某个准确率数字本身，而是它相对基线提高了多少。  
What matters is not a raw accuracy number by itself, but how much it improves over the baseline.

In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["DummyClassifier", "LogisticRegression"],
        "accuracy": [dummy_acc, acc],
    }
)

print(comparison)

In [ ]:
# 练习 1 / Exercise 1
# 请实现一个函数 build_logreg_pipeline()。
# Implement build_logreg_pipeline().
#
# 要求 / Requirements:
# 1. 返回一个 Pipeline / return a Pipeline
# 2. 第一层是 StandardScaler / first step is StandardScaler
# 3. 第二层是 LogisticRegression(max_iter=500) / second step is LogisticRegression(max_iter=500)

def build_logreg_pipeline():
    # TODO
    pass


# test_pipe = build_logreg_pipeline()
# print(test_pipe)

In [ ]:
# 练习 1 参考答案 / Exercise 1 Reference Solution

def build_logreg_pipeline_solution():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=500)),
        ]
    )


print(build_logreg_pipeline_solution())

In [ ]:
# 练习 2 / Exercise 2
# 用一句话回答 / Answer in one sentence:
# 为什么标准化器不能先在全量数据上 fit，再切分训练集和测试集？
# Why should we not fit the scaler on the full dataset before splitting into train and test?

参考回答 / Reference answer:

如果标准化器先看到了全量数据，就等于让训练流程提前接触了测试集信息，这会导致数据泄漏 / data leakage，并让评估结果过于乐观。  
If the scaler is fit on the full dataset first, the training process indirectly sees information from the test set, causing data leakage and overly optimistic evaluation.

## 7. 小结 / Summary

这份 notebook 的关键不是某个具体模型，而是完整流程。  
The key lesson here is not a specific model, but the complete workflow.

你现在应该能回答 / You should now be able to answer:

1. 为什么先切分再标准化？ / Why do we split before scaling?
2. 为什么要先做基线模型？ / Why do we build a baseline model first?
3. 混淆矩阵能补充什么信息？ / What extra information does a confusion matrix provide?
4. `Pipeline` 为什么更安全、更清晰？ / Why is a `Pipeline` safer and cleaner?

下一步建议 / Suggested next step:

- `Phase 0` 基础已经形成闭环，接下来可以进入 `PyTorch` 的 `Tensor` 基础 / Phase 0 now forms a closed loop, so the next natural step is `PyTorch` tensor basics.